# Run pyCIAM for Inequality Analysis

> **Original Reference:** This notebook adapts the original
> [`run-pyCIAM-slrquantiles.ipynb`](https://gitlab.com/ClimateImpactLab/coastal/projects/pyciam/-/blob/inequality/notebooks/models/run-pyCIAM-slrquantiles.ipynb)
> from the [pyCIAM inequality branch](https://gitlab.com/ClimateImpactLab/coastal/projects/pyciam/-/tree/inequality).
> It updates paths and dependencies to work with the current regional SCC infrastructure
> and produce coastal damage estimates for the inequality analysis.

This notebook runs pyCIAM with temperature-limit SLR scenarios to produce
coastal damage estimates for the inequality analysis.

**Prerequisites:**
- Run `01_process_slr_inputs.ipynb` first to create the processed SLR data

**Inputs:**
- Processed SLR: `PATH_SLR_INEQUALITY` (from step 01)
- SLIIDERS: `PATH_SLIIDERS` (reused from regional SCC)
- Surge lookup: `PATHS_SURGE_LOOKUP` (reused from regional SCC)
- Params: `params.json`

**Output:**
- `pyCIAM_outputs_inequality_1000_ssp234.zarr` with dimensions:
  - `case`: 2 (noAdaptation, optimalfixed)
  - `costtype`: 6 (wetland, inundation, relocation, protection, stormCapital, stormPopulation)
  - `gadmid`: ~6000 regions
  - `scenario`: 6 (ncc_ar6, tlim1.5, tlim2.0, tlim3.0, tlim4.0, tlim5.0)
  - `sample`: 1000
  - `year`: 2 (2050, 2090)
  - `ssp`: 3 (SSP2, SSP3, SSP4)
  - `iam`: 2 (IIASA GDP, OECD Env-Growth)

In [ ]:
import sys
#sys.path.append("../..")

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
from collections import OrderedDict
from itertools import product
from cloudpathlib import AnyPath

from config import (
    # Paths
    PATH_PARAMS,
    PATH_SLIIDERS,
    PATH_SLIIDERS_SEG,
    PATH_SLR_INEQUALITY,
    PATH_REFA_INEQUALITY,
    PATHS_SURGE_LOOKUP,
    PATH_OUTPUT_TMP,
    PATH_OUTPUT_INEQUALITY,
    PATH_OUTPUT_FINAL,
    DIR_SCRATCH,
    # Parameters
    SEG_VAR,
    ADM_VAR,
    MC_DIM,
    SEG_CHUNKSIZE,
    REFA_SEG_CHUNKSIZE,
    SAMPLE_CHUNKSIZE,
    OUTPUT_YEARS,
    OUTPUT_SSPS,
    OUTPUT_CASES,
    N_SAMPLES_TOTAL,
    N_WORKERS_MIN,
    N_WORKERS_MAX,
    # Metadata
    AUTHOR,
    CONTACT,
    HISTORY,
    STORAGE_OPTIONS,
    # Functions
    save_zarr,
)

In [ ]:
# === TEST MODE ===
TEST_MODE = True

if TEST_MODE:
    from config import *
    N_SAMPLES_TOTAL = 5
    N_WORKERS_MIN = 2
    N_WORKERS_MAX = 4
    SEG_CHUNKSIZE = 1
    SAMPLE_CHUNKSIZE = 5
    PATH_SLR_INEQUALITY = DIR_SCRATCH / "test-ar6-tlim-slr.zarr"
    PATH_OUTPUT_TMP = DIR_SCRATCH / "test-pyciam-inequality-tmp.zarr"
    PATH_OUTPUT_INEQUALITY = DIR_SCRATCH / "test-pyciam-inequality-output.zarr"
    PATH_OUTPUT_FINAL = DIR_SCRATCH / "test-pyciam-inequality-final.zarr"
    PATH_REFA_INEQUALITY = DIR_SCRATCH / "test-refa-inequality.zarr"
    PATH_SLIIDERS_SEG = DIR_SCRATCH / "test-sliiders-seg-inequality.zarr"


In [ ]:
# Import pyCIAM components
from pyCIAM.constants import CASE_DICT, CASES, COSTTYPES, SOLVCASES
from pyCIAM.io import (
    check_finished_zarr_workflow,
    create_template_dataarray,
    load_ciam_inputs,
)
from pyCIAM.run import (
    calc_all_cases,
    get_refA,
    optimize_case,
)
from pyCIAM.utils import (
    add_attrs_to_result,
    collapse_econ_inputs_to_seg,
    subset_econ_inputs,
)

## Configuration

In [ ]:
# Run parameters
OVERWRITE = False  # Set True to regenerate existing outputs

DESCRIPTION = "Projected coastal damages from pyCIAM for inequality analysis, using temperature-limit SLR scenarios."

# Convert paths to AnyPath
params_path = AnyPath(PATH_PARAMS)
econ_input_path = str(PATH_SLIIDERS)
econ_input_path_seg = PATH_SLIIDERS_SEG
slr_input_paths = [PATH_SLR_INEQUALITY]
slr_names = ["ar6"]
refA_path = PATH_REFA_INEQUALITY
surge_input_paths = {k: AnyPath(v) for k, v in PATHS_SURGE_LOOKUP.items()}
output_path = PATH_OUTPUT_TMP

# Quantiles = sample indices (1 to 1000)
quantiles = np.arange(1, N_SAMPLES_TOTAL + 1)

In [ ]:
# Read model parameters
params = pd.read_json(params_path)["values"]
print("Model parameters:")
print(f"  Discount rate: {params.dr}")
print(f"  Move factor: {params.movefactor}")
print(f"  Planning periods: {params.at_start}")

## Setup Dask Cluster

In [ ]:
import os
from dask_gateway import Gateway
from distributed.diagnostics.plugin import PipInstall

img = os.environ.get("JUPYTER_IMAGE", None)

gateway = Gateway()
cluster = gateway.new_cluster(
    idle_timeout=900,
    profile="micro",
    **(dict(worker_image=img, scheduler_image=img) if img else {})
)

client = cluster.get_client()

# Install required packages on workers
pip_installer = PipInstall(
    packages=["cloudpathlib==0.13.0", "rhg_compute_tools"],
)
client.register_worker_plugin(pip_installer)
client.run_on_scheduler(pip_installer.install)

cluster.adapt(minimum=N_WORKERS_MIN, maximum=N_WORKERS_MAX)
cluster

## Step 1: Collapse SLIIDERS to Segment Level (if needed)

In [ ]:
if OVERWRITE or not econ_input_path_seg.is_dir():
    print("Collapsing SLIIDERS to segment level...")
    collapse_econ_inputs_to_seg(
        econ_input_path,
        econ_input_path_seg,
        seg_var_subset=None,
        output_chunksize=100,
        storage_options=STORAGE_OPTIONS,
        seg_var=SEG_VAR,
    )
    print("Done.")
else:
    print(f"Segment-level SLIIDERS already exists at {econ_input_path_seg}")

## Step 2: Load Economic Inputs and Define Output Template

In [ ]:
# Load economic inputs
ciam_in = subset_econ_inputs(
    xr.open_zarr(str(econ_input_path), chunks=None, storage_options=STORAGE_OPTIONS),
    SEG_VAR,
    seg_var_subset=None,
)

# Filter to years around output years (2040-2060 for 2050, 2080-2100 for 2090)
ciam_in = ciam_in.sel(year=np.concatenate((
    np.arange(2040, 2060),
    np.arange(2080, 2100)
)))

print(f"Economic inputs loaded: {len(ciam_in[SEG_VAR])} segment-regions")

In [ ]:
# Load SLR to get scenario names
slr_test = xr.open_zarr(str(slr_input_paths[0]), chunks=None)
scenarios = slr_test.scenario.values
print(f"SLR scenarios: {scenarios}")

In [ ]:
# Create output template
attr_dict = {
    "updated": pd.Timestamp.now(tz="US/Pacific").strftime("%c"),
    "planning_period_start_years": params.at_start,
    "author": AUTHOR,
    "contact": CONTACT,
    "description": DESCRIPTION,
    "history": HISTORY,
}

coords = OrderedDict(
    {
        "case": CASES,
        "costtype": COSTTYPES,
        SEG_VAR: ciam_in[SEG_VAR].values,
        "scenario": scenarios,
        "sample": quantiles,
        "year": np.arange(params.model_start, ciam_in.year.max().item() + 1),
        **{dim: ciam_in[dim].values for dim in ["ssp", "iam"] if dim in ciam_in.dims},
    }
)

chunks = {SEG_VAR: 1, "case": len(coords["case"]) - 1}
chunks = {k: -1 if k not in chunks else chunks[k] for k in coords}

# Create template
out_ds = create_template_dataarray(coords.keys(), coords, chunks).to_dataset(name="costs")
out_ds["npv"] = out_ds.costs.isel(year=0, costtype=0, drop=True).astype("float64")
out_ds["optimal_case"] = out_ds.npv.isel(case=0, drop=True).astype("uint8")
out_ds.attrs.update(attr_dict)
out_ds = add_attrs_to_result(out_ds)

print(f"Output template: {out_ds.dims}")

In [ ]:
# Save template
if OVERWRITE or not output_path.is_dir():
    out_ds.to_zarr(
        str(output_path),
        compute=False,
        mode="w",
        storage_options=STORAGE_OPTIONS,
    )
    print(f"Output template saved to {output_path}")

## Step 3: Calculate Reference Adaptation Heights (refA)

Reference adaptation heights are calculated under a no-climate-change scenario.

In [ ]:
if OVERWRITE or not refA_path.is_dir():
    print("Calculating reference adaptation heights...")
    
    segs = np.unique(ciam_in.seg)
    seg_grps = [
        segs[i : i + REFA_SEG_CHUNKSIZE]
        for i in range(0, len(segs), REFA_SEG_CHUNKSIZE)
    ]
    
    samps = np.arange(1, N_SAMPLES_TOTAL + 1)
    samp_grps = [
        samps[i : i + SAMPLE_CHUNKSIZE]
        for i in range(0, len(samps), SAMPLE_CHUNKSIZE)
    ]
    
    grps = list(product(seg_grps, samp_grps))
    print(f"Processing {len(grps)} refA groups...")
    
    refa_futs = client.map(
        get_refA,
        grps,
        output_path=str(refA_path),
        econ_input_path=econ_input_path_seg,
        slr_input_path=slr_input_paths[0],
        params=params,
        surge_input_path=surge_input_paths["seg"],
        mc_dim=MC_DIM,
        storage_options=STORAGE_OPTIONS,
        quantiles=quantiles,
        diaz_inputs=False,
        eps=1,
    )
    
    # Wait for completion
    from distributed import wait
    wait(refa_futs)
    print("Reference adaptation heights calculated.")
else:
    print(f"Reference adaptation heights already exist at {refA_path}")

## Step 4: Run pyCIAM Cost Calculations

Calculate costs for all adaptation cases across all segment-regions and samples.

In [ ]:
# Create groups for parallel processing
groups = [
    ciam_in[SEG_VAR].isel({SEG_VAR: slice(i, i + SEG_CHUNKSIZE)}).values
    for i in np.arange(0, len(ciam_in[SEG_VAR]), SEG_CHUNKSIZE)
]

samps = np.arange(1, N_SAMPLES_TOTAL + 1)
samp_grps = [
    samps[i : i + SAMPLE_CHUNKSIZE]
    for i in range(0, len(samps), SAMPLE_CHUNKSIZE)
]

# Create all combinations
grps = list(product(groups, samp_grps))
print(f"Total groups to process: {len(grps)}")

In [ ]:
# Run Stage 1: Calculate costs for all adaptation cases
print("Running pyCIAM cost calculations...")

ciam_futs = np.array(
    client.map(
        calc_all_cases,
        grps,
        params=params,
        econ_input_path=econ_input_path,
        slr_input_paths=slr_input_paths,
        slr_names=slr_names,
        output_path=output_path,
        refA_path=refA_path,
        surge_input_path=surge_input_paths[SEG_VAR],
        seg_var=SEG_VAR,
        mc_dim=MC_DIM,
        quantiles=quantiles,
        storage_options=STORAGE_OPTIONS,
        diaz_inputs=False,
        check=False,
    )
)

print(f"Submitted {len(ciam_futs)} tasks")

In [ ]:
# Monitor progress
from distributed import wait, progress
progress(ciam_futs)

In [ ]:
# Wait for completion
wait(ciam_futs)

# Check for errors
n_errors = sum(1 for f in ciam_futs if f.status == 'error')
n_finished = sum(1 for f in ciam_futs if f.status == 'finished')
print(f"Finished: {n_finished}, Errors: {n_errors}")

## Step 5: Optimize Adaptation Cases

Select the optimal adaptation strategy for each segment.

In [ ]:
# Create sample groups for optimization
sample_ids = {}
for i in range(4):
    sample_ids[i] = np.arange(250 * i + 1, 250 * (i + 1) + 1)

# Create optimization futures
seg_adm_ser = pd.Series(ciam_in[SEG_VAR].values)
seg_adm_ser.index = ciam_in.seg.values
seg_grps = seg_adm_ser.groupby(seg_adm_ser.index).apply(list)

# ... (optimization logic as in original notebook)

## Step 6: Extract Final Output

Filter to the required cases, years, and SSPs, then aggregate to gadmid level.

In [ ]:
# Extract and filter output
print("Extracting final output...")

t = xr.open_zarr(str(output_path))
t = t.sel(
    case=OUTPUT_CASES,
    ssp=OUTPUT_SSPS,
    year=OUTPUT_YEARS,
)[["costs"]]

# Clear encodings
for v in t.data_vars:
    t[v].encoding.clear()
for k, v in t.coords.items():
    if v.dtype == object:
        t[k] = v.astype("unicode")

print(f"Filtered output: {t.dims}")

In [ ]:
# Aggregate from seg_ir to gadmid
print("Aggregating to gadmid level...")

out = xr.open_zarr(
    str(output_path),
    chunks={"case": -1, SEG_VAR: 2},
)
out = out.sel(case=OUTPUT_CASES, ssp=OUTPUT_SSPS, year=OUTPUT_YEARS)[["costs"]]

# Group by gadmid and sum
out["costs"] = (
    out.costs.groupby(ciam_in[ADM_VAR]).sum().chunk({ADM_VAR: 2})
).persist()

out = out.drop(SEG_VAR).unify_chunks()

# Clear encodings
for v in out.data_vars:
    out[v].encoding.clear()
for k, v in out.coords.items():
    if v.dtype == object:
        out[k] = v.astype("unicode")

out = out.persist()
print(f"Aggregated output: {out.dims}")

In [ ]:
# Save final output
print(f"Saving to {PATH_OUTPUT_FINAL}...")
out.to_zarr(str(PATH_OUTPUT_FINAL), storage_options=STORAGE_OPTIONS, mode="w")
print("Done!")

In [ ]:
# Verify output
print("Verifying output...")
verify = xr.open_zarr(str(PATH_OUTPUT_FINAL))
print(verify)

# Check for missing values in optimalfixed
assert (
    verify.sel(case='optimalfixed', drop=True)
    .sum(dim='costtype')
    .costs.notnull()
    .all()
)
print("All values present!")

In [ ]:
# Cleanup
client.close()
cluster.close()